<a href="https://colab.research.google.com/github/taonanm/Climate-Risk-Loss-Insurance-Model/blob/main/ClimateRiskLossModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import glob

files = glob.glob("/content/drive/MyDrive/Climate Risk and Insurance Loss Modeling/StormEvents_details-ftp_v1.0_d*_c*.csv")

dfs = [pd.read_csv(f, low_memory=False) for f in files]

noaa = pd.concat(dfs, ignore_index=True)

print(noaa.shape)
noaa.head()

(641907, 54)


,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,damage_property,damage_crops,total_damage
0,202503,31,1104,202503,31,1106,201366,1252415,GEORGIA,13,...,33.4757,-85.238,33.4757,-85.238,A cold-front initiated a line of thunderstorms...,Tree down at the intersection of highway 5 and...,CSV,1000.0,0.0,1000.0
1,202503,30,1552,202503,30,1555,200337,1241136,MICHIGAN,26,...,41.7900,-86.100,41.8200,-86.070,A cold front pushed into the area during the a...,A brief EF-1 tornado was confirmed in Edwardsb...,CSV,100000.0,0.0,100000.0
2,202501,5,1800,202501,6,2227,197733,1222851,VIRGINIA,51,...,NaN,NaN,NaN,NaN,An area of low pressure tracked across souther...,NaN,CSV,0.0,0.0,0.0
3,202501,3,1300,202501,3,1900,197761,1223112,MARYLAND,24,...,NaN,NaN,NaN,NaN,An area of low pressure moved off into New Eng...,NaN,CSV,0.0,0.0,0.0
4,202501,3,1300,202501,3,1900,197761,1223113,MARYLAND,24,...,NaN,NaN,NaN,NaN,An area of low pressure moved off into New Eng...,NaN,CSV,0.0,0.0,0.0


In [ ]:
import pandas as pd
import glob

files = glob.glob("/content/drive/MyDrive/Climate Risk and Insurance Loss Modeling/StormEvents_details-ftp_v1.0_d*_c*.csv")

dfs = [pd.read_csv(f, low_memory=False) for f in files]

noaa = pd.concat(dfs, ignore_index=True)

def convert_damage(val):
    if pd.isna(val):
        return 0
    val = str(val).strip()

    if val.endswith('K'):
        return float(val[:-1]) * 1_000
    elif val.endswith('M'):
        return float(val[:-1]) * 1_000_000
    elif val.endswith('B'):
        return float(val[:-1]) * 1_000_000_000
    else:
        try:
            return float(val)
        except:
            return 0

noaa['damage_property'] = noaa['DAMAGE_PROPERTY'].apply(convert_damage)
noaa['damage_crops'] = noaa['DAMAGE_CROPS'].apply(convert_damage)

noaa['total_damage'] = noaa['damage_property'] + noaa['damage_crops']

In [9]:
noaa['BEGIN_DATE_TIME_STR'] = noaa['BEGIN_YEARMONTH'].astype(str) + \
                              noaa['BEGIN_DAY'].astype(str).str.zfill(2) + \
                              noaa['BEGIN_TIME'].astype(str).str.zfill(4)

noaa['BEGIN_DATE_TIME'] = pd.to_datetime(noaa['BEGIN_DATE_TIME_STR'], format='%Y%m%d%H%M', errors='coerce')

noaa['year'] = noaa['BEGIN_DATE_TIME'].dt.year

noaa.drop(columns=['BEGIN_DATE_TIME_STR'], inplace=True)

major_events = ['Hurricane', 'Flood', 'Tornado', 'Storm Surge', 'Wildfire']
noaa = noaa[noaa['EVENT_TYPE'].isin(major_events)]

noaa_grouped = noaa.groupby(['STATE','year']).agg({
    'total_damage':'sum',
    'EVENT_TYPE':'count'
}).rename(columns={'EVENT_TYPE':'event_count'}).reset_index()

noaa_grouped.head()

,STATE,year,total_damage,event_count
0,ALABAMA,2015,9036000.0,56
1,ALABAMA,2016,2018000.0,93
2,ALABAMA,2017,2242000.0,92
3,ALABAMA,2018,381000.0,83
4,ALABAMA,2019,270000.0,168


In [5]:
noaa_grouped.describe()

,year,total_damage,event_count
count,526.000000,5.260000e+02,526.000000
mean,2019.598859,1.076401e+08,87.692015
std,3.024459,9.022188e+08,103.034327
min,2015.000000,0.000000e+00,1.000000
25%,2017.000000,2.380000e+05,23.000000
50%,2019.500000,3.375500e+06,58.500000
75%,2022.000000,1.762085e+07,117.750000
max,2025.000000,1.856723e+10,880.000000


In [11]:
zillow = pd.read_csv("/content/drive/MyDrive/Climate Risk and Insurance Loss Modeling/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")

zillow.columns

Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS', '2000-01-31',
       ...
       '2025-05-31', '2025-06-30', '2025-07-31', '2025-08-31', '2025-09-30',
       '2025-10-31', '2025-11-30', '2025-12-31', '2026-01-31', '2026-02-28'],
      dtype='object', length=323)

In [17]:
zillow['STATE'] = zillow['StateName']

zillow_long = zillow.melt( id_vars=['STATE'], var_name='date', value_name='home_value' )

zillow_long['date'] = pd.to_datetime(zillow_long['date'], format='%Y-%m-%d', errors='coerce')

zillow_long['year'] = zillow_long['date'].dt.year

zillow_long['year'] = zillow_long['year'].astype('Int64')

zillow_state = zillow_long.groupby(['STATE','year'])['home_value'].mean().reset_index()

zillow_state.head()

,STATE,year,home_value
0,AK,2000,149404.43824
1,AK,2001,173764.832924
2,AK,2002,190635.648868
3,AK,2003,200341.445803
4,AK,2004,214403.076595


In [8]:
import pandas as pd

zillow = pd.read_csv("/content/drive/MyDrive/Climate Risk and Insurance Loss Modeling/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")

zillow['STATE'] = zillow['StateName']

zillow_long = zillow.melt(
    id_vars=['STATE'],
    var_name='date',
    value_name='home_value'
)

zillow_long['date'] = pd.to_datetime(zillow_long['date'], format='%Y-%m-%d', errors='coerce')

zillow_long['year'] = zillow_long['date'].dt.year

zillow_long['year'] = zillow_long['year'].astype('Int64')

zillow_state = zillow_long.groupby(['STATE','year'])['home_value'].mean().reset_index()

zillow_state.head()

,STATE,year,home_value
0,AK,2000,149404.43824
1,AK,2001,173764.832924
2,AK,2002,190635.648868
3,AK,2003,200341.445803
4,AK,2004,214403.076595


In [9]:
zillow_state.describe()

,year
count,1377.0
mean,2013.0
std,7.791711
min,2000.0
25%,2006.0
50%,2013.0
75%,2020.0
max,2026.0


In [10]:
pop = pd.read_csv("/content/drive/MyDrive/Climate Risk and Insurance Loss Modeling/DECENNIALDHC2020.P1-2026-03-26T162154.csv")
housing = pd.read_csv("/content/drive/MyDrive/Climate Risk and Insurance Loss Modeling/DECENNIALDHC2020.H1-2026-03-26T162235.csv")


In [11]:
pop.head()

,Label (Grouping),Alabama,Alaska,Arizona,Arkansas,California,Colorado,Connecticut,Delaware,District of Columbia,...,Tennessee,Texas,Utah,Vermont,Virginia,Washington,West Virginia,Wisconsin,Wyoming,Puerto Rico
0,Total,"5,024,279","733,391","7,151,502","3,011,524","39,538,223","5,773,714","3,605,944","989,948","689,545",...,"6,910,840","29,145,505","3,271,616","643,077","8,631,393","7,705,281","1,793,716","5,893,718","576,851","3,285,874"


In [12]:
housing.head()

,Label (Grouping),Alabama,Alaska,Arizona,Arkansas,California,Colorado,Connecticut,Delaware,District of Columbia,...,Tennessee,Texas,Utah,Vermont,Virginia,Washington,West Virginia,Wisconsin,Wyoming,Puerto Rico
0,Total,"2,288,330","326,200","3,082,000","1,365,265","14,392,140","2,491,404","1,530,197","448,735","350,364",...,"3,031,605","11,589,324","1,151,414","334,318","3,618,247","3,202,241","855,635","2,727,726","271,887","1,598,159"


In [13]:
pop_t = pop.T.reset_index()
pop_t.columns = ['STATE', 'population']
pop_t = pop_t.iloc[1:]

housing_t = housing.T.reset_index()
housing_t.columns = ['STATE', 'housing_units']
housing_t = housing_t.iloc[1:]

census = pd.merge(pop_t, housing_t, on='STATE')

census['STATE'] = census['STATE'].str.upper().str.strip()

census['population'] = census['population'].astype(str).str.replace(',', '', regex=False).str.strip()
census['housing_units'] = census['housing_units'].astype(str).str.replace(',', '', regex=False).str.strip()

census['population'] = pd.to_numeric(census['population'], errors='coerce')
census['housing_units'] = pd.to_numeric(census['housing_units'], errors='coerce')

census.head()

,STATE,population,housing_units
0,ALABAMA,5024279,2288330
1,ALASKA,733391,326200
2,ARIZONA,7151502,3082000
3,ARKANSAS,3011524,1365265
4,CALIFORNIA,39538223,14392140


In [14]:
fema = pd.read_csv("/content/drive/MyDrive/Climate Risk and Insurance Loss Modeling/DisasterDeclarationsSummaries.csv")
fema.columns

Index(['femaDeclarationString', 'disasterNumber', 'state', 'declarationType',
       'declarationDate', 'fyDeclared', 'incidentType', 'declarationTitle',
       'ihProgramDeclared', 'iaProgramDeclared', 'paProgramDeclared',
       'hmProgramDeclared', 'incidentBeginDate', 'incidentEndDate',
       'disasterCloseoutDate', 'tribalRequest', 'fipsStateCode',
       'fipsCountyCode', 'placeCode', 'designatedArea',
       'declarationRequestNumber', 'lastIAFilingDate', 'incidentId', 'region',
       'designatedIncidentTypes', 'lastRefresh', 'hash', 'id'],
      dtype='object')

In [15]:
fema['year'] = pd.to_datetime(fema['declarationDate'], errors='coerce').dt.year

fema_grouped = fema.groupby(['state', 'year']).size().reset_index(name='disaster_count')

fema_grouped.rename(columns={'state': 'STATE'}, inplace=True)

fema_grouped['STATE'] = fema_grouped['STATE'].str.upper().str.strip()

fema_grouped.head()

,STATE,year,disaster_count
0,AK,1953,1
1,AK,1954,1
2,AK,1955,1
3,AK,1964,1
4,AK,1967,1


In [16]:
state_map = {
    'AL': 'ALABAMA','AK': 'ALASKA','AZ': 'ARIZONA','AR': 'ARKANSAS','CA': 'CALIFORNIA',
    'CO': 'COLORADO','CT': 'CONNECTICUT','DE': 'DELAWARE','FL': 'FLORIDA','GA': 'GEORGIA',
    'HI': 'HAWAII','ID': 'IDAHO','IL': 'ILLINOIS','IN': 'INDIANA','IA': 'IOWA',
    'KS': 'KANSAS','KY': 'KENTUCKY','LA': 'LOUISIANA','ME': 'MAINE','MD': 'MARYLAND',
    'MA': 'MASSACHUSETTS','MI': 'MICHIGAN','MN': 'MINNESOTA','MS': 'MISSISSIPPI',
    'MO': 'MISSOURI','MT': 'MONTANA','NE': 'NEBRASKA','NV': 'NEVADA','NH': 'NEW HAMPSHIRE',
    'NJ': 'NEW JERSEY','NM': 'NEW MEXICO','NY': 'NEW YORK','NC': 'NORTH CAROLINA',
    'ND': 'NORTH DAKOTA','OH': 'OHIO','OK': 'OKLAHOMA','OR': 'OREGON','PA': 'PENNSYLVANIA',
    'RI': 'RHODE ISLAND','SC': 'SOUTH CAROLINA','SD': 'SOUTH DAKOTA','TN': 'TENNESSEE',
    'TX': 'TEXAS','UT': 'UTAH','VT': 'VERMONT','VA': 'VIRGINIA','WA': 'WASHINGTON',
    'WV': 'WEST VIRGINIA','WI': 'WISCONSIN','WY': 'WYOMING'
}

zillow_state['STATE'] = zillow_state['STATE'].map(state_map)

fema_grouped['STATE'] = fema_grouped['STATE'].map(state_map)


In [17]:
merged = noaa_grouped.merge(
    zillow_state,
    on=['STATE','year'],
    how='left'
)

merged = merged.merge(
    fema_grouped,
    on=['STATE','year'],
    how='left'
)

merged = merged.merge(
    census,
    on='STATE',
    how='left'
)

merged['STATE'] = merged['STATE'].str.upper().str.strip()

merged['disaster_count'] = merged['disaster_count'].fillna(0)

merged.head()


,STATE,year,total_damage,event_count,home_value,disaster_count,population,housing_units
0,ALABAMA,2015,9036000.0,56,119521.781211,0.0,5024279.0,2288330.0
1,ALABAMA,2016,2018000.0,93,120566.091096,39.0,5024279.0,2288330.0
2,ALABAMA,2017,2242000.0,92,123490.21805,116.0,5024279.0,2288330.0
3,ALABAMA,2018,381000.0,83,127527.91027,27.0,5024279.0,2288330.0
4,ALABAMA,2019,270000.0,168,134066.310986,13.0,5024279.0,2288330.0


In [18]:
merged['exposure'] = merged['home_value'] * merged['housing_units']

merged['damage_per_capita'] = merged['total_damage'] / merged['population']

merged['risk_score'] = (
    merged['total_damage'] * 0.5 +
    merged['event_count'] * 0.2 +
    merged['disaster_count'] * 0.1 +
    merged['exposure'] * 0.2
)

merged[['STATE','year','risk_score']].head()

,STATE,year,risk_score
0,ALABAMA,2015,54705573530.709801
1,ALABAMA,2016,55180009670.269547
2,ALABAMA,2017,56518395163.84436
3,ALABAMA,2018,58365379101.037766
4,ALABAMA,2019,61357727318.721443


In [19]:
merged['norm_damage'] = merged['total_damage'] / merged['total_damage'].max()
merged['norm_events'] = merged['event_count'] / merged['event_count'].max()
merged['norm_fema'] = merged['disaster_count'] / merged['disaster_count'].max()
merged['norm_exposure'] = merged['exposure'] / merged['exposure'].max()

merged['risk_score_norm'] = (
    merged['norm_damage'] * 0.4 +
    merged['norm_events'] * 0.2 +
    merged['norm_fema'] * 0.1 +
    merged['norm_exposure'] * 0.3
)

merged[['STATE','year','risk_score','risk_score_norm']].head()

,STATE,year,risk_score,risk_score_norm
0,ALABAMA,2015,54705573530.709801,0.022187
1,ALABAMA,2016,55180009670.269547,0.036786
2,ALABAMA,2017,56518395163.84436,0.049149
3,ALABAMA,2018,58365379101.037766,0.033091
4,ALABAMA,2019,61357727318.721443,0.050666


In [20]:
ml_data = merged[['norm_damage','norm_events','norm_fema','norm_exposure','risk_score_norm']].dropna()

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

X = ml_data[['norm_damage','norm_events','norm_fema','norm_exposure']]
y = ml_data['risk_score_norm']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("R2:", r2)

MSE: 7.457633157585836e-05
R2: 0.9561495835880895


In [22]:
merged = merged.sort_values(['STATE','year'])

merged['next_risk'] = merged.groupby('STATE')['risk_score_norm'].shift(-1)

ml_data = merged[['norm_damage','norm_events','norm_fema','norm_exposure','next_risk']].dropna()

X = ml_data[['norm_damage','norm_events','norm_fema','norm_exposure']]
y = ml_data['next_risk']

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 0.0013470537863658058
R2: 0.6766760068702768


In [24]:
importance = pd.Series(model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False))

norm_exposure    0.764969
norm_events      0.141487
norm_damage      0.061650
norm_fema        0.031894
dtype: float64


In [25]:
merged.to_csv("climate_risk_final.csv", index=False)

from google.colab import files
files.download("climate_risk_final.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>